# 1. 인간 피드백 기반 강화학습 (RLHF)과 DPO (Direct Preference Optimization)

이전 차시까지 우리는 입력문장에 부합하는 대답을 생성하도록 정밀 학습을 시키는 파인튜닝(Full FT, LoRA) 기법을 실습했습니다.
하지만 AI가 유용하고(Helpful), 정직하며(Honest), 무해한(Harmless) 답변을 제공하도록 즉, **인간의 가치와 선호에 정렬(Alignment)** 되도록 하는 것은 단순한 분류나 텍스트 복제 방식의 학습으로는 달성하기 어렵습니다.

 LLM의 정렬을 위해 필수적으로 쓰이는 **RLHF** 와 최신 정렬 기법인 **DPO** 의 개념을 이해하고, 실습용 `trl` 라이브러리를 활용하여 GPT-2 모델을 직접 선호 정렬 학습시킵니다.

## 1. RLHF와 DPO의 핵심 이해

### 1) RLHF (Reinforcement Learning from Human Feedback) 란?
- 모델의 출력을 인간의 평가 기준에 부합하도록 지도하는 3단계 정렬 파이프라인입니다.
  1. **SFT (Supervised Fine-tuning)**: 지시어와 원하는 고품질 답변 데이터로 지도학습을 진행합니다.
  2. **RM (Reward Model, 보상 모델) 학습**: 동일한 질문에 대한 여러 답변을 인간이 선호도에 따라 순위를 매기고, 이를 바탕으로 질문-답변의 점수를 예측하는 보상 모델을 학습시킵니다.
  3. **PPO (Proximal Policy Optimization) 강화학습**: 2단계에서 만든 보상 모델의 점수를 극대화하도록 PPO 알고리즘을 사용해 LLM의 가중치를 갱신합니다.
- **한계**: 보상 모델 학습 및 PPO 강화학습을 결합하는 루프가 복잡하고, 학습이 매우 불안정하며 메모리 소모가 극도로 심합니다.

### 2) DPO (Direct Preference Optimization) 란?
- Stanford 대학 연구진이 제안한 기법으로, **별도의 보상 모델(Reward Model)과 복잡한 강화학습 PPO 과정 없이** , 사람이 선호한 답변(Chosen)과 거부한 답변(Rejected)만을 가지고 **수학적 손실 함수를 사용해 직접 정책 모델을 튜닝** 하는 방식입니다.
- 수학적 치환을 통해 RLHF의 복잡한 PPO 목적 함수를 단순한 이진 교차 엔트로피(Binary Cross-Entropy) 손실 함수로 매핑하여 학습 안전성과 효율성을 대폭 끌어올렸습니다.

In [1]:
from datasets import Dataset
from trl import DPOTrainer,DPOConfig
from peft import LoraConfig, get_peft_model, TaskType
import time
import torch
import pandas as pd
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForCausalLM,
    Seq2SeqTrainer,Seq2SeqTrainingArguments,DataCollatorForSeq2Seq
    )
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'using devcies : {device}')

using devcies : cpu


## 2. DPO용 가상 선호 데이터셋 정의
DPO 학습 데이터셋은 반드시 아래 세 가지 열을 갖춰야 합니다:
1. `prompt`: 모델에 전달하는 질문/지시문
2. `chosen`: 사람이 선호하는 긍정 답변 (Target Winner)
3. `rejected`: 사람이 선호하지 않는 잘못되거나 부정적인 답변 (Target Loser)

In [2]:
dpo_train_data = [
    {
        "prompt": "Review: This movie is a masterpiece. Sentiment: ",
        "chosen": "positive",
        "rejected": "negative"
    },
    {
        "prompt": "Review: I hated this film. Total waste of time. Sentiment: ",
        "chosen": "negative",
        "rejected": "positive"
    },
    {
        "prompt": "Review: Outstanding acting and brilliant plot. Sentiment: ",
        "chosen": "positive",
        "rejected": "negative"
    },
    {
        "prompt": "Review: Terrible acting and boring screenplay. Sentiment: ",
        "chosen": "negative",
        "rejected": "positive"

    }
]

dpo_test_data = [
    {
        "prompt": "Review: The movie was mediocre, but the ending was spectacular! Sentiment: ",
        "chosen": "positive",
        "rejected": "negative"
    },
    {
        "prompt": "Review: Worst movie I have seen this year. Avoid at all costs. Sentiment: ",
        "chosen": "negative",
        "rejected": "positive"
    }
]

train_dataset = Dataset.from_list(dpo_train_data)
test_dataset = Dataset.from_list(dpo_test_data)
print("DPO dataset initialized!")

DPO dataset initialized!


## 3. 베이스 모델 로드 및 LoRA 설정 적용
- DPO 학습 대상 모델로 디코더 형태의 모델인 `gpt2`를 로드합니다.
- 학습 파라미터 소모를 대폭 줄이기 위해 LoRA 기법을 결합하여 어댑터만 튜닝하도록 구성합니다.

### AutoModelForSeq2SeqLM, AutoModelForCausalLM 비교
```
- Encoder-Decoder               Decoder Only
-   T5, BART                    GPT,LIma
- 입력 -> Encoder                입력->Decoder 
- 벡터 -> Decoder- 출력           -> 다른 토큰으로 예측 -> 다른 토큰으로 예측
```

In [3]:
import torch
device = "cuda" if torch.cuda.is_available() else 'cpu'
model = 'gpt2'
tokenizer = AutoTokenizer.from_pretrained(model)
model = AutoModelForCausalLM.from_pretrained(model)

# gpt-2 패딩 설정 조정
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = model.config.eos_token_id
model.to(device)

# LoRA 설정
peft_config= LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r = 8,          # 저차원 랭크의 크기
    lora_alpha=32,  # 가중치 스케일 상수
    lora_dropout=0.1,
    target_modules=['c_attn'],
)

# PEFT 모델로 변환
model = get_peft_model(model, peft_config=peft_config)
# 파라메터 계산함수
def get_trainable_params(model):
    all_param = 0
    trainable_params= 0
    for _, param in model.named_parameters():
        all_param += param.numel()  # 파라메터 개수
        if param.requires_grad:
            trainable_params += param.numel()
    return all_param,trainable_params, trainable_params/all_param*100

all_p, train_p, pct = get_trainable_params(model)
all_p, train_p, pct

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Exception in thread Thread-5 (_readerthread):
Traceback (most recent call last):
  File "c:\Users\Playdata\miniconda3\envs\torch_env\lib\threading.py", line 1016, in _bootstrap_inner
    self.run()
  File "c:\Users\Playdata\miniconda3\envs\torch_env\lib\threading.py", line 953, in run
    self._target(*self._args, **self._kwargs)
  File "c:\Users\Playdata\miniconda3\envs\torch_env\lib\subprocess.py", line 1515, in _readerthread
    buffer.append(fh.read())
  File "c:\Users\Playdata\miniconda3\envs\torch_env\lib\codecs.py", line 322, in decode
    (result, consumed) = self._buffer_decode(data, self.errors, final)
UnicodeDecodeError: 'utf-8' codec can't decode byte 0xc0 in position 6: invalid start byte
c:\Users\Playdata\miniconda3\envs\torch_env\lib\site-packages\peft\tuners\lora\layer.py:2504: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


(124734720, 294912, 0.23643136409814366)

## 4. DPO 학습 설정 및 Trainer 구동
- `DPOConfig` 클래스를 선언하고 `DPOTrainer`에 로드하여 학습을 진행합니다.
- `beta`는 DPO의 중요 매개변수로, 기준 참조 모델(Reference Model)과의 KL 패널티 계수를 나타냅니다. (보통 0.1~0.5 사용)

In [4]:
train_args = DPOConfig(
    output_dir = './gpt_dpo_results',
    eval_strategy='epoch',
    learning_rate=5e-4,  # LoRA DPO용 다소 높은 학습률
    num_train_epochs=2,
    beta = 0.1,
    use_cpu = True
)

trainer = DPOTrainer(
    model=model,
    ref_model=None,  # 메모리 보존
    args = train_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    processing_class=tokenizer
)
start_time = time.time() 
trainer.train()
training_time = time.time() - start_time
print(f'DPO training  completed : {training_time:.2f} seconds') 

Adding EOS to train dataset:   0%|          | 0/4 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/4 [00:00<?, ? examples/s]

[RANK 0] Mismatch between tokenized prompt and the start of tokenized prompt+chosen. This may be due to unexpected tokenizer behavior, whitespace issues, or special token handling. Verify that the tokenizer is processing text consistently.
[RANK 0] Mismatch between tokenized prompt and the start of tokenized prompt+rejected. This may be due to unexpected tokenizer behavior, whitespace issues, or special token handling. Verify that the tokenizer is processing text consistently.
[RANK 0] Mismatch between tokenized prompt and the start of tokenized prompt+chosen. This may be due to unexpected tokenizer behavior, whitespace issues, or special token handling. Verify that the tokenizer is processing text consistently.
[RANK 0] Mismatch between tokenized prompt and the start of tokenized prompt+rejected. This may be due to unexpected tokenizer behavior, whitespace issues, or special token handling. Verify that the tokenizer is processing text consistently.
[RANK 0] Mismatch between tokenized 

Adding EOS to eval dataset:   0%|          | 0/2 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/2 [00:00<?, ? examples/s]

[RANK 0] Mismatch between tokenized prompt and the start of tokenized prompt+chosen. This may be due to unexpected tokenizer behavior, whitespace issues, or special token handling. Verify that the tokenizer is processing text consistently.
[RANK 0] Mismatch between tokenized prompt and the start of tokenized prompt+rejected. This may be due to unexpected tokenizer behavior, whitespace issues, or special token handling. Verify that the tokenizer is processing text consistently.
[RANK 0] Mismatch between tokenized prompt and the start of tokenized prompt+chosen. This may be due to unexpected tokenizer behavior, whitespace issues, or special token handling. Verify that the tokenizer is processing text consistently.
[RANK 0] Mismatch between tokenized prompt and the start of tokenized prompt+rejected. This may be due to unexpected tokenizer behavior, whitespace issues, or special token handling. Verify that the tokenizer is processing text consistently.
[transformers] The tokenizer has new

Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Logits/chosen,Logits/rejected,Mean Token Accuracy,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected
1,No log,0.693147,2.973538,116.000000,-79.045593,-79.045593,0.000000,0.011996,0.011996,0.000000,0.000000,-7.094398,-7.094398
2,No log,0.693147,2.792102,232.000000,-78.981140,-78.981140,0.000000,-0.006105,-0.006105,0.000000,0.000000,-7.275404,-7.275404


DPO training  completed : 507.96 seconds


## 5. DPO 정렬 모델 결과 검증 및 평가
학습이 끝난 모델에 테스트용 리뷰 데이터를 주입해 보아 긍정/부정 선호 정렬이 올바르게 나타나는지 확인합니다.

In [5]:
def generate_prediction(prompt, model, tokenizer, max_new_tokens=10):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=max_new_tokens)
    # 생성된 텍스트 중 프롬프트 부분을 제외한 새로 추가된 토큰만 해독
    generated_ids = outputs[0][inputs["input_ids"].shape[1]:]
    pred_text = tokenizer.decode(generated_ids, skip_special_tokens=True).strip().lower()
    return pred_text

print("\n--- Testing DPO Aligned Model ---")
for item in dpo_test_data:
    prompt = item["prompt"]
    true_chosen = item["chosen"]
    pred = generate_prediction(prompt, model, tokenizer)
    print(f"Prompt: {prompt}")
    print(f"-> Expected (Chosen): {true_chosen} | Model Output: {pred}\n")


--- Testing DPO Aligned Model ---
Prompt: Review: The movie was mediocre, but the ending was spectacular! Sentiment: 
-> Expected (Chosen): positive | Model Output: i'm not sure if it's because of

Prompt: Review: Worst movie I have seen this year. Avoid at all costs. Sentiment: 
-> Expected (Chosen): negative | Model Output: i'm not a fan of the movie,



### Supervised Fine-Tuning(SFT) 학습
- DPO 모델이 Review: .... Sentiment:... 뒤에 단답형 답변 positive negative를 붙여서 완료하는 규칙

In [9]:
sft_train_list = [
    {'text':f"{item['prompt']}{item['chosen']}{tokenizer.eos_token}"}
for item in dpo_train_data
]
sft_dataset = Dataset.from_list(sft_train_list)
from trl import SFTTrainer, SFTConfig
sft_config = SFTConfig(
    output_dir="./gpt2_sft_results",
    eval_strategy="no",
    learning_rate=1e-4,
    per_device_train_batch_size=2,
    num_train_epochs=12,      # 데이터가 극소량이므로 형식을 인지하도록 에폭을 늘립니다.
    max_length=64,
    use_cpu=(device == "cpu"),
    report_to="none"
)

sft_trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=sft_dataset,
    processing_class=tokenizer
)

sft_start_time = time.time()
sft_trainer.train()
sft_time = time.time() - sft_start_time
print(f"SFT completed in {sft_time:.2f} seconds.")

Adding EOS to train dataset:   0%|          | 0/4 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/4 [00:00<?, ? examples/s]

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
10,4.268370
20,4.008657


SFT completed in 921.62 seconds.


- SFT를 통해 출력 형식을 터득한 모델 가중치를 기반으로, 긍정 평가는 긍정 단어로, 부정 평가는 부정 단어로 밀어내는 **선호 정렬 DPO 학습** 을 수행합니다.

In [12]:
training_args = DPOConfig(
    output_dir="./gpt2_dpo_results",
    eval_strategy="epoch",
    learning_rate=2e-4,       # SFT 모델을 기반으로 하므로 적절한 학습률 적용
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    num_train_epochs=12,
    beta=0.1,                 # DPO margin
    max_length=64,
    use_cpu=(device == "cpu"),
    report_to="none"
)

trainer = DPOTrainer(
    model=model,
    ref_model=None,           # PEFT 활성 상태이므로 None으로 주면 베이스 모델(SFT 미수행)을 ref로 참조
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    processing_class=tokenizer
)

dpo_start_time = time.time()
trainer.train()
dpo_time = time.time() - dpo_start_time
print(f"DPO completed in {dpo_time:.2f} seconds.")

Adding EOS to train dataset:   0%|          | 0/4 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/4 [00:00<?, ? examples/s]

[RANK 0] Mismatch between tokenized prompt and the start of tokenized prompt+chosen. This may be due to unexpected tokenizer behavior, whitespace issues, or special token handling. Verify that the tokenizer is processing text consistently.
[RANK 0] Mismatch between tokenized prompt and the start of tokenized prompt+rejected. This may be due to unexpected tokenizer behavior, whitespace issues, or special token handling. Verify that the tokenizer is processing text consistently.
[RANK 0] Mismatch between tokenized prompt and the start of tokenized prompt+chosen. This may be due to unexpected tokenizer behavior, whitespace issues, or special token handling. Verify that the tokenizer is processing text consistently.
[RANK 0] Mismatch between tokenized prompt and the start of tokenized prompt+rejected. This may be due to unexpected tokenizer behavior, whitespace issues, or special token handling. Verify that the tokenizer is processing text consistently.
[RANK 0] Mismatch between tokenized 

Adding EOS to eval dataset:   0%|          | 0/2 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/2 [00:00<?, ? examples/s]

[RANK 0] Mismatch between tokenized prompt and the start of tokenized prompt+chosen. This may be due to unexpected tokenizer behavior, whitespace issues, or special token handling. Verify that the tokenizer is processing text consistently.
[RANK 0] Mismatch between tokenized prompt and the start of tokenized prompt+rejected. This may be due to unexpected tokenizer behavior, whitespace issues, or special token handling. Verify that the tokenizer is processing text consistently.
[RANK 0] Mismatch between tokenized prompt and the start of tokenized prompt+chosen. This may be due to unexpected tokenizer behavior, whitespace issues, or special token handling. Verify that the tokenizer is processing text consistently.
[RANK 0] Mismatch between tokenized prompt and the start of tokenized prompt+rejected. This may be due to unexpected tokenizer behavior, whitespace issues, or special token handling. Verify that the tokenizer is processing text consistently.


Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Logits/chosen,Logits/rejected,Mean Token Accuracy,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected
1,No log,0.693147,3.551903,116.000000,-78.937210,-78.937210,0.000000,0.001805,0.001805,0.000000,0.000000,-6.678328,-6.678328
2,No log,0.693147,3.531960,232.000000,-79.074493,-79.074493,0.000000,0.002403,0.002403,0.000000,0.000000,-6.672344,-6.672344
3,No log,0.693147,3.412733,348.000000,-78.817535,-78.817535,0.000000,0.008757,0.008757,0.000000,0.000000,-6.608805,-6.608805
4,No log,0.693147,3.429529,464.000000,-79.072693,-79.072693,0.000000,-0.002455,-0.002455,0.000000,0.000000,-6.720926,-6.720926
5,0.693147,0.693147,3.394068,580.000000,-78.988426,-78.988426,0.000000,-0.002229,-0.002229,0.000000,0.000000,-6.718668,-6.718668
6,0.693147,0.693147,3.244978,696.000000,-79.110519,-79.110519,0.000000,-0.019296,-0.019296,0.000000,0.000000,-6.889328,-6.889328
7,0.693147,0.693147,3.439703,812.000000,-78.937103,-78.937103,0.000000,-0.001213,-0.001213,0.000000,0.000000,-6.708506,-6.708506
8,0.693147,0.693147,3.448786,928.000000,-78.929016,-78.929016,0.000000,-0.003200,-0.003200,0.000000,0.000000,-6.728371,-6.728371
9,0.693147,0.693147,3.408487,1044.000000,-78.847198,-78.847198,0.000000,-0.003784,-0.003784,0.000000,0.000000,-6.734215,-6.734215
10,0.693147,0.693147,3.412472,1160.000000,-78.725128,-78.725128,0.000000,-0.001560,-0.001560,0.000000,0.000000,-6.711969,-6.711969


DPO completed in 3352.04 seconds.


In [13]:
print("\n--- Inference after DPO (Final Alignment Check) ---")
for item in dpo_test_data:
    prompt = item["prompt"]
    true_chosen = item["chosen"]
    pred = generate_prediction(prompt, model, tokenizer)
    print(f"Prompt: {prompt}")
    print(f"-> Expected (Chosen): {true_chosen} | Model Output: {pred}\n")


--- Inference after DPO (Final Alignment Check) ---
Prompt: Review: The movie was mediocre, but the ending was spectacular! Sentiment: 
-> Expected (Chosen): positive | Model Output: i'm not sure if it's because of

Prompt: Review: Worst movie I have seen this year. Avoid at all costs. Sentiment: 
-> Expected (Chosen): negative | Model Output: i'm not a fan of the movie,

